In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from PROPS_EV.calculateEVS import *
from BACKTEST.backtest import *
from MODELS.pipeline import *

### Load Model

In [2]:
PTSmodel = joblib.load('Models/xgbPTSModelNoOppStats26.pkl')
PTSfeatures = joblib.load('Models/topPTSfeaturesNoOppStats26.pkl')

### Load Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  

s25= pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv('../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_20251021.csv')
singlePTSBookies = usData[usData['CATEGORY'] == 'player_points']

dfsData = pd.read_csv('../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_20251021.csv')

### Top EVs for single bets

In [4]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['ODDS'] <= 200) & (usData['ODDS'] >= -200)]

results = single_bet(
    data=s25,
    bookmakers=singlePTSBookies,
    model=PTSmodel,
    features=PTSfeatures,
    edge_threshold=5.0,
    stake=100,
    simulations=10000, 
    std_window=15,
    min_std=1.5,
    max_std=8.0,
    stat_col='PTS'
)
singleBets = results.sort_values(by='EV%', ascending=False).reset_index(drop=True)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets_{today}.csv', index=False)

Processing single bets...


## Top EVs for 2 leg bets

### Underdog picks

In [5]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = prizepickspairsEV(
    data=s25,
    bookmakers=dfsPTS,
    model=PTSmodel,
    features=PTSfeatures,
    edge_threshold=5,
    stake=100,
    simulations=1000,
    std_window=15,
    min_std=1.5,
    max_std=8.0,
    stat_col='PTS'
)
underdogPairs = results.sort_values(by='EV%', ascending=False).reset_index(drop=True)
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdog_{today}.csv', index=False)

Processing pairs...


### Prizepicks picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = prizepickspairsEV(
    data=s25,
    bookmakers=dfsPTS,
    model=PTSmodel,
    features=PTSfeatures,
    edge_threshold=5,
    stake=100,
    simulations=1000,
    std_window=15,
    min_std=1.5,
    max_std=8.0,
    stat_col='PTS'
)
pairsPrizepicks = results.sort_values(by='EV%', ascending=False).reset_index(drop=True)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicks_{today}.csv', index=False)

Processing pairs...


## 3 leg parlay

### Underdog picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = prizepicks3LegEV(
    data=s25,
    bookmakers=dfsPTS,
    model=PTSmodel,
    features=PTSfeatures,
    edge_threshold=5,
    stake=100,
    simulations=10000,
    std_window=15,
    min_std=1.5,
    max_std=8.0,
    stat_col='PTS'
)
underdogTrios = threeLeg.sort_values(by='EV%', ascending=False).reset_index(drop=True)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios_{today}.csv', index=False)

Processing 3-leg parlays...


### Prizepicks picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = prizepicks3LegEV(
    data=s25,
    bookmakers=dfsPTS,
    model=PTSmodel,
    features=PTSfeatures,
    edge_threshold=5,
    stake=100,
    simulations=10000,
    std_window=15,
    min_std=1.5,
    max_std=8.0,
    stat_col='PTS'
)
triosPrizepicks = threeLeg.sort_values(by='EV%', ascending=False).reset_index(drop=True)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios_{today}.csv', index=False)

Processing 3-leg parlays...
